# Tutorial 1 — Factorize

This tutorial covers the `scads_drvi.factorize.model` API — reading a fit record,
inspecting and repairing a checkpoint, computing gate responsibilities — and how those
pieces feed the one object a fit's results now live in, `scads_drvi.io.result`'s
`embed`:

- Reading a fit record with `fit_meta()` — from a `fit.meta.json`, or from a finished
  result h5ad
- Inspecting checkpoint keys and detecting the `torch.compile` prefix
- Repairing a compile-prefixed checkpoint with `repair_compile_prefix()`
- `load_kwargs()` for multi-GPU jobs
- `split_responsibility()` for gated models
- Building the canonical `AnnData` with `build_embed()` and writing it with
  `write_result()`

We use a synthetic 3,000-cell plant dataset throughout. The checkpoint cells need real
PyTorch tensors and run as shown; `build_embed()` needs a trained model, so there we
stand in with a small class exposing only the two methods it calls
(`get_latent_representation`, `set_latent_dimension_stats`) — the same stand-in
`tests/test_portability.py` uses. The real `load_fit()` / `latent()` calls (shown in the
final markdown cell) require `scvi-tools >= 1.5` and the real DRVI model class from
`drvi-py`.

## 0. Shared setup — synthetic plant dataset

In [ ]:
import json
import tempfile
from pathlib import Path

import h5py
import numpy as np
import torch

TISSUES = ("leaf", "root", "stem", "flower")
LINEAGES = ("mesophyll", "epidermis", "vasculature")
FINE_TYPES = {
    "mesophyll": ("meso_1", "meso_2", "meso_3"),
    "epidermis": ("epi_1", "epi_2"),
    "vasculature": ("vasc_1", "vasc_2", "vasc_3"),
}
CULTIVARS = ("cv_alpha", "cv_beta", "cv_gamma")
TRAITS = ("plant_height", "grain_yield")
N_CELLS, N_FACTORS, N_KEPT = 3_000, 12, 9

root = Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)

fine = [t for lin in LINEAGES for t in FINE_TYPES[lin]]
lineage_of = {t: lin for lin in LINEAGES for t in FINE_TYPES[lin]}
cell_ids = np.array([f"plate{i % 5}:bc{i:05d}".encode() for i in range(N_CELLS)])
fine_codes = rng.integers(0, len(fine), size=N_CELLS).astype(np.int32)
tissue_codes = rng.integers(0, len(TISSUES), size=N_CELLS).astype(np.int32)
cult_codes = rng.integers(0, len(CULTIVARS), size=N_CELLS).astype(np.int32)
reads = rng.integers(500, 200_000, size=N_CELLS).astype(np.int64)

obs_path = root / "matrix.h5ad"
with h5py.File(obs_path, "w") as fh:
    obs = fh.create_group("obs")
    obs.attrs["_index"] = "cell_uid"
    obs.attrs["encoding-type"] = "dataframe"
    obs.create_dataset("cell_uid", data=cell_ids)
    for name, codes, cats in (
        ("fine_type", fine_codes, fine),
        ("tissue", tissue_codes, TISSUES),
        ("cultivar", cult_codes, CULTIVARS),
    ):
        g = obs.create_group(name)
        g.create_dataset("categories", data=np.array([c.encode() for c in cats]))
        g.create_dataset("codes", data=codes)
    obs.create_dataset("total_reads", data=reads)
    lin_codes = np.array(
        [LINEAGES.index(lineage_of[fine[c]]) for c in fine_codes], dtype=np.int32
    )
    g = obs.create_group("lineage")
    g.create_dataset("categories", data=np.array([lin.encode() for lin in LINEAGES]))
    g.create_dataset("codes", data=lin_codes)

fit_dir = root / "fits" / "assembly_v3"
fit_dir.mkdir(parents=True, exist_ok=True)
print("Working directory:", root)

## 1. Write `fit.meta.json` and read it back with `fit_meta()`

`fit_meta()` now takes an explicit path: a `fit.meta.json` file (or the directory that
contains it), or a finished result h5ad — it is auto-detected from the `.h5ad` suffix,
and in that case the record comes from `uns["provenance"]` rather than a standalone
file. The only required key is `n_latent`; the rest have defaults or are inferred.

In [ ]:
from scads_drvi.factorize.model import fit_meta

meta_record = {
    "n_latent": N_FACTORS,
    "n_split_latent": 4,
    "batch_key": "cultivar",
    "min_fragment": None,
    "depth_col": "total_reads",
    "n_cells": N_CELLS,
    "n_features": 50_000,
    "gene_likelihood": "zinb",
    "dispersion": "gene",
    "seed": 42,
    "smoke_test": False,
}
(fit_dir / "fit.meta.json").write_text(json.dumps(meta_record, indent=2))

meta = fit_meta(fit_dir)  # or fit_meta(fit_dir / "fit.meta.json")
print("n_latent:      ", meta.n_latent)
print("n_split_latent:", meta.n_split_latent)
print("batch_key:     ", meta.batch_key)
print("is_gated:      ", meta.is_gated)  # False — no min_fragment set
print("n_cells:       ", meta.n_cells)
print("n_features:    ", meta.n_features)

## 2. Create a minimal checkpoint and inspect its keys

`checkpoint_keys()` reads key names from a saved `.pt` file without instantiating the
model — useful for validation on a login node where loading the full network would OOM.

We create a synthetic state dict that mirrors the structure of a real DRVI checkpoint.

In [ ]:
from scads_drvi.factorize.model import COMPILE_PREFIX, checkpoint_keys, has_compile_prefix

model_dir = fit_dir / "model"
model_dir.mkdir(exist_ok=True)

# Synthetic state dict — same key structure as a real DRVI checkpoint
state_dict = {
    "encoder.layer.0.weight": torch.randn(64, 50_000),
    "encoder.layer.0.bias": torch.zeros(64),
    "z_mean_encoder.weight": torch.randn(N_FACTORS, 64),
    "z_mean_encoder.bias": torch.zeros(N_FACTORS),
    "decoder.layer.0.weight": torch.randn(64, N_FACTORS),
    "decoder.layer.0.bias": torch.zeros(64),
}
torch.save({"model_state_dict": state_dict}, model_dir / "model.pt")

keys = checkpoint_keys(model_dir)
print(f"Checkpoint has {len(keys)} keys:")
for k in keys:
    print("  ", k)

print("\nhas_compile_prefix:", has_compile_prefix(model_dir))

## 3. Detect and repair a `torch.compile` prefix

When a model is saved after `torch.compile()` every key acquires an `_orig_mod.` prefix.
The checkpoint cannot be loaded without the wrapper class; `repair_compile_prefix`
writes a fixed copy to a separate directory (never overwrites the original).

In [ ]:
from scads_drvi.factorize.model import repair_compile_prefix

# Create a checkpoint that looks like it was saved from a compiled module
compiled_dir = fit_dir / "model_compiled"
compiled_dir.mkdir(exist_ok=True)
compiled_sd = {f"{COMPILE_PREFIX}{k}": v for k, v in state_dict.items()}
torch.save({"model_state_dict": compiled_sd}, compiled_dir / "model.pt")

print("Before repair — first two keys:")
for k in checkpoint_keys(compiled_dir)[:2]:
    print("  ", k)
print("has_compile_prefix:", has_compile_prefix(compiled_dir))

# Repair: strip the prefix and write to a new directory
repaired_dir = fit_dir / "model_repaired"
repair_compile_prefix(compiled_dir, out_dir=repaired_dir)

print("\nAfter repair — first two keys:")
for k in checkpoint_keys(repaired_dir)[:2]:
    print("  ", k)
print("has_compile_prefix:", has_compile_prefix(repaired_dir))

## 4. `load_kwargs()` — device pinning for multi-GPU jobs

In a `torchrun` job each rank must load its shard of the model onto its own GPU.
`load_kwargs(rank)` returns the right map-location kwargs for `DRVI.load()`. With
`rank=None` it defers to `"cuda"` if a GPU is available or `"cpu"` otherwise.

In [ ]:
from scads_drvi.factorize.model import load_kwargs

kwargs = load_kwargs(rank=None)
print("load_kwargs (single-process):", kwargs)

# In a real torchrun job:
# from scads_drvi.factorize.multigpu import init
# r = init()
# kwargs = load_kwargs(r.rank)
# model = DRVI.load(repaired_dir, adata=adata, **kwargs)

## 5. `split_responsibility()` — gate weights for gated models

Split-latent DRVI trains a two-expert gate: each cell's contribution to a split
dimension is controlled by a Softmax over two logits. `split_responsibility` converts
those logits to responsibility weights in [0, 1].

The `is_gated` flag on `FitMeta` tells you whether the model was trained with gating.

In [ ]:
from scads_drvi.factorize.model import split_responsibility

n_split = meta.n_split_latent  # 4 in our synthetic meta
logits = torch.randn(N_CELLS, n_split, 2)

resp = split_responsibility(logits)
# Output shape equals input shape: softmax is over split_dim=-2 (n_splits),
# so each (cell, feature) position holds responsibility weights across splits.
print("logits shape:        ", tuple(logits.shape))  # (3000, 4, 2)
print("responsibility shape:", resp.shape)  # (3000, 4, 2)
print(f"range: [{float(resp.min()):.3f}, {float(resp.max()):.3f}]")
# Sum across splits must equal 1 for every (cell, feature) pair
print("split weights for cell 0, feature 0 (sum to 1):", resp[0, :, 0].round(4))

## 6. Build the result `AnnData` with `build_embed()`

The fit's results now live in one `AnnData` — `obs` = cells, `var` = one row per latent
dimension, `X` = the signed latent representation — built by `build_embed()` and
written by `write_result()` (see `scads_drvi.io.result`). This replaces the
`topic_loadings.npz` / `topic_loadings.tsv` contract this tutorial used to read back
with `read_loadings()`.

`build_embed()` takes a trained model and calls the model's own
`set_latent_dimension_stats()` to populate `embed.var` with DRVI's `vanished` /
`order` / `title` columns — no vanished-dimension detection is reimplemented here. In a
real analysis `model` is `scads_drvi.factorize.model.load_fit(model_dir, adata=adata)`;
since we have no trained checkpoint in this tutorial, we stand in with a tiny class
exposing only the two methods `build_embed` needs.

In [ ]:
import anndata as ad

from scads_drvi.io.artifacts import cell_metadata
from scads_drvi.io.result import build_embed, read_result, write_result


class FakeDRVIModel:
    """Stands in for a trained ``scvi.external.DRVI``. In a real analysis this is
    ``scads_drvi.factorize.model.load_fit(model_dir, adata=adata)``; only the two
    methods ``build_embed`` calls are needed here — ``get_latent_representation`` and
    DRVI's own ``set_latent_dimension_stats``.
    """

    def __init__(self, z, vanished):
        self._z = z
        self._vanished = np.asarray(vanished, dtype=bool)

    def get_latent_representation(self, adata=None, indices=None, batch_size=128):
        out = self._z
        if indices is not None:
            out = out[np.asarray(indices)]
        return out

    def set_latent_dimension_stats(
        self, embed, adata=None, datamodule=None, vanished_threshold=0.5
    ):
        # A real DRVI model derives these from the fit itself; fixed here so the
        # tutorial is reproducible.
        embed.var["vanished"] = self._vanished
        embed.var["order"] = np.arange(embed.n_vars)
        embed.var["title"] = [f"DR {i + 1}" for i in range(embed.n_vars)]


obs_frame = cell_metadata(
    obs_path, columns=["fine_type", "tissue", "cultivar", "lineage", "total_reads"]
)
fake_adata = ad.AnnData(X=np.zeros((N_CELLS, 1), dtype=np.float32), obs=obs_frame)

z = rng.normal(0.2, 0.6, size=(N_CELLS, N_FACTORS)).astype(np.float32)
vanished = np.array([False] * N_KEPT + [True] * (N_FACTORS - N_KEPT))
model = FakeDRVIModel(z, vanished)

embed = build_embed(model, fake_adata, obs_columns=list(obs_frame.columns))
print("embed shape (cells x factors):", embed.shape)
embed.var[["vanished", "order", "title"]]

In [ ]:
fit_path = root / "assembly_v3.h5ad"
write_result(fit_path, embed, provenance={**meta_record, "fit_name": "assembly_v3"})
print("Wrote:", fit_path)

reread = read_result(fit_path)
print("n_obs, n_vars:", reread.shape)
print("provenance n_latent:", reread.uns["provenance"]["n_latent"])

# fit_meta() reads straight from a result h5ad too
meta_from_h5ad = fit_meta(fit_path)
print("fit_meta(fit_path).n_latent:", meta_from_h5ad.n_latent)

## Calling `load_fit()` and `latent()` with a real model

Everything above runs without a trained DRVI checkpoint. With a real one, the full
sequence — checkpoint through the finished result h5ad — is:

```python
import anndata as ad
from scads_drvi.factorize.model import (
    fit_meta, has_compile_prefix, repair_compile_prefix,
    load_fit, setup_anndata_like, latent,
)
from scads_drvi.io.result import build_embed, write_result

meta = fit_meta(fit_dir)                      # or fit_meta(fit_dir / "fit.meta.json")
adata = ad.read_h5ad(obs_path, backed="r")
setup_anndata_like(adata, meta)               # registers obs columns the model expects

model_dir = fit_dir / "model"
if has_compile_prefix(model_dir):
    model_dir = repair_compile_prefix(model_dir, out_dir=fit_dir / "model_repaired")

model = load_fit(model_dir, adata=adata, rank=None, device="cpu")

z = latent(model, adata=adata, indices=None, batch_size=4096)
# z: np.ndarray of shape (n_cells, n_latent)

embed = build_embed(
    model, adata, obs_columns=["fine_type", "tissue", "cultivar", "lineage"]
)
write_result(fit_dir.parent / "assembly_v3.h5ad", embed, provenance=meta.raw)
```

Requires `scvi-tools >= 1.5`, the matching DRVI model class, and `drvi-py`.